<a href="https://colab.research.google.com/github/grasht/projects_ML_HW_4/blob/main/HW4_pt1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1

In [ ]:
import kagglehub
import os

# Download TinyShakespeare from Kaggle
path = kagglehub.dataset_download("thedevastator/the-bards-best-a-character-modeling-dataset")

print("Path to dataset files:", path)

os.listdir(path)


Using Colab cache for faster access to the 'the-bards-best-a-character-modeling-dataset' dataset.
Path to dataset files: /kaggle/input/the-bards-best-a-character-modeling-dataset


['validation.csv', 'train.csv', 'test.csv']

In [ ]:
import torch

#Read the text in
with open(f'{path}/train.csv', 'r', encoding='utf-8') as f:
    train_data = f.read()

with open(f'{path}/test.csv', 'r', encoding='utf-8') as f:
    test_data = f.read()

#Put the characters in a sorted list so the ID's don't change between runs
chars = sorted(list(set(train_data)))
vocab_size = len(chars)

#Encoding and Decoding dictionaires
stoi = {ch:i for i,ch in enumerate(chars)} #str to int
itos = {i:ch for ch,i in stoi.items()} #int to str

def encode(s):
    return [stoi[c] for c in s]

def decode(i):
    return [itos[n] for n in i]

train_ids = torch.tensor(encode(train_data), dtype=torch.long)
test_ids  = torch.tensor(encode(test_data), dtype=torch.long)



In [ ]:
from torch.utils.data import Dataset

block_size = 128

class CharDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data) - block_size

    def __getitem__(self, idx):
      x = self.data[idx:idx+block_size]
      y = self.data[idx+1:idx+block_size+1]
      return x, y

train_dataset = CharDataset(train_ids)
test_dataset  = CharDataset(test_ids)

In [ ]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-mini'
model_config.vocab_size = vocab_size
model_config.block_size = block_size

model = GPT(model_config)

number of parameters: 2.71M


In [ ]:
# create a Trainer object
from mingpt.trainer import Trainer

batch_size = 64

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4
train_config.max_iters = 5000
train_config.num_workers = 0
train_config.batch_size = batch_size
trainer = Trainer(train_config, model, train_dataset)

def batch_end_callback(trainer):
    if trainer.iter_num % 50 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms. iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")

trainer.set_callback('on_batch_end', batch_end_callback)
trainer.run()

running on device cuda
iter_dt 0.00ms. iter 0: train loss 4.20443
iter_dt 24.08ms. iter 50: train loss 2.55059
iter_dt 24.06ms. iter 100: train loss 2.44118
iter_dt 24.02ms. iter 150: train loss 2.40712
iter_dt 27.66ms. iter 200: train loss 2.37745
iter_dt 23.93ms. iter 250: train loss 2.28461
iter_dt 23.41ms. iter 300: train loss 2.23211
iter_dt 24.03ms. iter 350: train loss 2.16947
iter_dt 24.13ms. iter 400: train loss 2.16138
iter_dt 23.84ms. iter 450: train loss 2.02995
iter_dt 25.12ms. iter 500: train loss 1.98840
iter_dt 23.90ms. iter 550: train loss 1.96355
iter_dt 24.05ms. iter 600: train loss 1.89851
iter_dt 24.11ms. iter 650: train loss 1.84292
iter_dt 24.06ms. iter 700: train loss 1.82218
iter_dt 20.35ms. iter 750: train loss 1.85423
iter_dt 23.93ms. iter 800: train loss 1.75779
iter_dt 24.12ms. iter 850: train loss 1.70721
iter_dt 24.11ms. iter 900: train loss 1.72449
iter_dt 24.00ms. iter 950: train loss 1.65517
iter_dt 23.82ms. iter 1000: train loss 1.65842
iter_dt 23.92m

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# now let's perform some evaluation
model = model.to(device)
model.eval();

cuda


In [ ]:
prompt = 'Such a tragedy to behold'
context = torch.tensor(encode(prompt), dtype=torch.long)[None, :].to(device)
generated = model.generate(context, max_new_tokens=600, temperature=0.8)

print(''.join([itos[int(i)] for i in generated[0]]))

Such a tragedy to behold the crown,
And then the senate of the people's son,
Which we have seen the sea of the sea,
Which we may see the sea of the sea,
Which we may see the sea of the senate,
Which the senate of the senate of the prince,
Which the senate of the prince to the prisoner.

Second Murderer:
What is the present shall be the present soul?

Second Murderer:
What is the present shall be the present soul?

Second Murderer:
What is the present shall be the present soul?

Second Murderer:
What is the present shall be the present soul?

Second Murderer:
What is the present shall be the present soul?

Second Murd


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device.")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
      x  = self.embed(x)
      out, hidden = self.lstm(x, hidden)
      logits = self.fc(out)
      return logits, hidden


Using cuda device.


In [ ]:
embedding_dim = 64
hidden_size = 128
num_layers = 2

modelLSTM = CharLSTM(vocab_size, embedding_dim, hidden_size, num_layers).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()

num_epochs = 500

for epoch in range(num_epochs):
  hidden = None
  total_loss = 0
  for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()

    logits, hidden = modelLSTM(x, hidden)
    # Detach hidden to prevent backprop through entire history
    hidden = tuple([h.detach() for h in hidden])

    # Reshape logits and targets to (batch*seq_len, vocab_size)
    loss = criterion(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

print(f"Epoch {epoch+1}/{num_epochs} - Loss: {total_loss/len(train_loader):.4f}")

RuntimeError: Expected hidden[0] size (2, 22, 128), got [2, 64, 128]